<a href="https://colab.research.google.com/github/Vaibhavi1007/Fake_Job_detectionAnd_analysis/blob/main/notebooks/sanika/01_data_cleaning_and_baseline_sanika.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Mount Drive

In [3]:
!unzip -o "/content/drive/MyDrive/HireTrust/archive.zip" -d /content/data

Archive:  /content/drive/MyDrive/HireTrust/archive.zip
  inflating: /content/data/fake_job_postings.csv  


Unzip the Dataset

In [4]:
!ls /content/data

fake_job_postings.csv


In [5]:
import pandas as pd

df = pd.read_csv("/content/data/fake_job_postings.csv")
print(df.shape)
print(df["fraudulent"].value_counts())
df.head()

(17880, 18)
fraudulent
0    17014
1      866
Name: count, dtype: int64


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


**How many posts are fake?**

This is the most important number in the project. Only about **4.84%** of posts are fake. That is called **class imbalance**.

**Why it matters:** a useless model that says "every post is real" would still be about 95% accurate, but it would catch zero scams. So we judge our models by **precision, recall and F1 on the fake class**, not by accuracy alone.

In [6]:
print(df["fraudulent"].mean() * 100)

4.8434004474272925


**Check missing values**

Many columns have blanks (salary range, department, and so on). This shows how many blanks each column has. This is normal for this dataset. We mainly care about the text columns.

In [7]:
print(df.isnull().sum())

job_id                     0
title                      0
location                 346
department             11547
salary_range           15012
company_profile         3308
description                1
requirements            2696
benefits                7212
telecommuting              0
has_company_logo           0
has_questions              0
employment_type         3471
required_experience     7050
required_education      8105
industry                4903
function                6455
fraudulent                 0
dtype: int64


**Combine the text columns into one**

The model reads one piece of text per post, so we join five text columns into a single column called `text`.

**Why fill blanks first:** if a value is missing (NaN), joining the columns would fail or give "nan". Replacing blanks with empty text avoids that.

**Why join the columns:** a scam can show up in any part of the post (title, company profile, description, requirements or benefits). Joining them lets the model see everything together.

In [8]:
text_cols = ["title", "company_profile", "description", "requirements", "benefits"]
df[text_cols] = df[text_cols].fillna("")
df["text"] = df[text_cols].agg(" ".join, axis=1)
df[["text", "fraudulent"]].head()

,text,fraudulent
0,"Marketing Intern We're Food52, and we've creat...",0
1,Customer Service - Cloud Video Production 90 S...,0
2,Commissioning Machinery Assistant (CMA) Valor ...,0
3,Account Executive - Washington DC Our passion ...,0
4,Bill Review Manager SpotSource Solutions LLC i...,0


**Clean the text**

We make the text uniform so the model treats the same word the same way:
- **Lowercase:** "Urgent" and "URGENT" and "urgent" become one word.
- **Remove links:** URLs are mostly noise for this model.
- **Remove symbols:** punctuation and special characters add noise.
- **Remove extra spaces:** keeps the text tidy.

In [9]:
import re

def clean_text(t):
    t = t.lower()
    t = re.sub(r"http\S+", " ", t)       # remove links
    t = re.sub(r"[^a-z0-9\s]", " ", t)   # remove symbols
    t = re.sub(r"\s+", " ", t).strip()   # remove extra spaces
    return t

df["text"] = df["text"].apply(clean_text)

**Check that the cleaning worked**

Print the first 300 characters of the first post. It should be lowercase, with no links or symbols.

In [10]:
df["text"].iloc[0][:300]

'marketing intern we re food52 and we ve created a groundbreaking and award winning cooking site we support connect and celebrate home cooks and give them everything they need in one place we have a top editorial business and engineering team we re focused on using technology to find new and better w'

**Save the clean data for the team**

We keep only the two columns the model needs: `text` and `fraudulent`. Gauri will load this file to train the models.

In [11]:
df[["text", "fraudulent"]].to_csv(
    "/content/drive/MyDrive/HireTrust/clean_jobs.csv", index=False)

## Summary
- Loaded about 17,880 job posts; only about 4.84% are fake (class imbalance).
- Combined five text columns into one `text` column and cleaned it.
- Saved `clean_jobs.csv` for the model-building step.